# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayesha-asif1/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
# --- Load data (same as w03_data_contract) ---
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=hf_token)

import pandas as pd
import numpy as np

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

# --- Filter to reliable rows ---
df_available = df_march[df_march['gsc_data_available'] == True].copy()

# --- Label ---
df_available['ctr'] = df_available['gsc_clicks'] / df_available['gsc_impressions']

# --- Features ---
df_available['day_of_week'] = pd.to_datetime(df_available['report_date']).dt.dayofweek
df_available['log_impressions'] = np.log1p(df_available['gsc_impressions'])

def position_bucket(pos):
    if pos <= 3:
        return 'top_3'
    elif pos <= 10:
        return 'top_10'
    elif pos <= 20:
        return 'top_20'
    else:
        return 'beyond_20'

df_available['position_bucket'] = df_available['gsc_avg_position'].apply(position_bucket)

# --- Final feature vector ---
feature_frame = df_available[[
    'client_hash_id', 'content_hash_id', 'report_date',   # context
    'ctr',                                                  # label
    'gsc_impressions', 'gsc_avg_position', 'day_of_week',   # raw features
    'log_impressions', 'position_bucket'                    # engineered features
]].copy()

print("Feature vector shape:", feature_frame.shape)
feature_frame.head()


Feature vector shape: (3611061, 9)


,client_hash_id,content_hash_id,report_date,ctr,gsc_impressions,gsc_avg_position,day_of_week,log_impressions,position_bucket
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0.000,20,3.350000,6,3.044522,top_10
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0.000,1,0.000000,6,0.693147,top_3
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,0.008,125,4.928000,6,4.836282,top_10
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0.000,7,4.000000,6,2.079442,top_10
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0.000,11,2.272727,6,2.484907,top_3


## 2. Feature notes (meaning, missing, categorical, available-when?)

## Feature notes

| Feature | Meaning | Missing values | Type | Available before prediction? |
|---|---|---|---|---|
| gsc_impressions | How many times the page was shown in search results | 0 missing | Numeric | Yes — known before any click occurs |
| gsc_avg_position | Page's average search ranking position that day | 0 missing | Numeric | Yes — ranking signal, not derived from clicks |
| day_of_week | Day of the week (0=Mon...6=Sun) from report_date | 0 missing | Categorical (numeric-coded) | Yes — date is always known in advance |
| log_impressions | log1p transform of gsc_impressions, to compress skewed distribution | 0 missing (derived from a complete column) | Numeric | Yes — pure math transform of an available feature |
| position_bucket | gsc_avg_position grouped into top_3 / top_10 / top_20 / beyond_20 | 0 missing | Categorical | Yes — direct categorization of an available feature |

No missing values were found in any of the source columns (gsc_avg_position, gsc_impressions, ctr) after filtering to gsc_data_available == True, so no imputation was required. If missing values appeared in the future, gsc_avg_position gaps would need explicit handling — a silent NaN would otherwise fall into the "beyond_20" bucket by default, which would be misleading.

In [7]:
# Missing values check
print("Missing gsc_avg_position:", feature_frame['gsc_avg_position'].isna().sum())
print("Missing gsc_impressions:", feature_frame['gsc_impressions'].isna().sum())
print("Missing ctr:", feature_frame['ctr'].isna().sum())
print()
print("position_bucket value counts:")
print(feature_frame['position_bucket'].value_counts(dropna=False))

Missing gsc_avg_position: 0
Missing gsc_impressions: 0
Missing ctr: 0

position_bucket value counts:
position_bucket
top_10       1456122
beyond_20     908354
top_3         727362
top_20        519223
Name: count, dtype: int64


## 3. The leakage hunt

### Result and honest interpretation

Correlation with CTR turned out weaker than expected — gsc_clicks showed only 0.10 correlation, and GA4/session columns were even weaker (0.001–0.07). This is because CTR is a ratio (clicks/impressions), not raw clicks, so a simple linear correlation doesn't fully capture the relationship.

However, low correlation does not mean no leakage. gsc_clicks remains excluded because it is mathematically part of the label's formula, not because of its correlation strength — a feature can be a direct leak even with a weak linear correlation, especially in combination with other fields. GA4/session columns remain excluded because they are recorded only after a click has already occurred, making them causally downstream of the outcome being predicted, regardless of their correlation strength.d proxy, even though it isn't a literal restatement of ctr.

In [8]:
# Reload full row set (with GA4 columns) to test correlation with CTR
suspect_cols = [
    'gsc_clicks', 'gsc_sum_position',
    'ga4_pageviews', 'ga4_sessions', 'ga4_users',
    'ga4_engaged_sessions', 'ga4_total_engagement_sec',
    'sessions_organic', 'sessions_direct', 'sessions_referral',
    'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events'
]

leakage_check = df_available[suspect_cols].copy()
leakage_check['ctr'] = df_available['ctr']

correlations = leakage_check.corr(numeric_only=True)['ctr'].drop('ctr').sort_values(ascending=False)
print("Correlation of candidate columns with CTR (label):")
print(correlations)

Correlation of candidate columns with CTR (label):
gsc_clicks                  0.103332
sessions_organic            0.073855
ga4_engaged_sessions        0.042393
ga4_total_engagement_sec    0.028790
ga4_pageviews               0.027938
ga4_sessions                0.023440
ga4_users                   0.022238
scroll_events               0.020030
sessions_direct             0.010824
sessions_social             0.007063
sessions_referral           0.005169
sessions_ai                 0.001812
sessions_paid               0.001243
gsc_sum_position           -0.008761
Name: ctr, dtype: float64


## 4. What I excluded and why

## What I excluded and why

| Excluded field | Reason |
|---|---|
| gsc_clicks | Directly used to compute the label (ctr = gsc_clicks / gsc_impressions). Including it as a feature would leak the answer into the model, regardless of its measured correlation strength. |
| gsc_sum_position | Redundant with gsc_avg_position (sum_position = avg_position × impressions) and harder to interpret on its own — no new information, added risk of confusion. |
| ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec | On-page engagement metrics that can only be recorded after a user has already clicked through from search. Causally downstream of the CTR outcome. |
| sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai | Session-source breakdowns, also only recorded after a click/visit has occurred. Same post-click timing issue as GA4 metrics above. |
| ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other | AI-referral breakdown columns, recorded after a visit — same reasoning, and additionally very sparse (most rows likely zero), which would add noise without helping this specific CTR prediction task. |
| scroll_events | On-page behavior metric, only exists after the user has landed on the page — post-click. |
| client_hash_id, content_hash_id, report_date | Not excluded from the dataset, but excluded as direct model *inputs* — used only for grouping, filtering, and identifying rows, not fed into the model as predictive features (they don't generalize — a model trained on specific ID values would overfit and be meaningless on new pages/clients). |
| gsc_data_available, ga4_data_available, client_has_gsc, client_has_ga4 | Used only as filters to select reliable rows (e.g., gsc_data_available == True), not as model inputs. |

In [9]:
all_columns = df_march.columns.tolist()
used_as_features = ['gsc_impressions', 'gsc_avg_position', 'day_of_week', 'log_impressions', 'position_bucket']
used_as_label_source = ['gsc_clicks', 'gsc_impressions']
used_as_context = ['client_hash_id', 'content_hash_id', 'report_date', 'gsc_data_available']

excluded = [c for c in all_columns if c not in used_as_features + used_as_label_source + used_as_context]
print(f"Total columns: {len(all_columns)}")
print(f"Explicitly excluded: {len(excluded)}")
print(excluded)

Total columns: 31
Explicitly excluded: 24
['client_has_gsc', 'client_has_ga4', 'ga4_data_available', 'gsc_sum_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.